# Model Implementation: Setting 1, default hyperparameters (Task 2)
**CM2604 Machine Learning, Part 01**

We implement the three required classifiers with scikit-learn **default hyperparameters** as a baseline. Tuning (Setting 2) is done in the next notebook, and the two settings are compared there.

Each model is a `Pipeline` of *preprocessor -> model*. Scaling and encoding are therefore re-fitted inside every cross-validation fold, which prevents data leakage.

In [1]:
import sys
sys.path.append("../src")

import os
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from preprocessing import load_and_clean, split_data, build_preprocessor, RANDOM_STATE
from evaluation import evaluate, results_table

os.makedirs("../results", exist_ok=True)

df = load_and_clean()
X_train, X_test, y_train, y_test = split_data(df)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (5634, 19) Test: (1409, 19)


## 1. Define the three models (defaults)
- **KNN**: classifies by the majority class of the k nearest customers (default k = 5). Distance-based, so scaling matters.
- **Logistic Regression**: a linear model that outputs a churn probability. `max_iter` is raised only so the solver converges.
- **Random Forest**: an ensemble of decision trees trained on random subsets, which captures non-linear relationships and interactions.

In [2]:
def make_pipeline(model):
    return Pipeline([("prep", build_preprocessor(X_train)), ("model", model)])

default_models = {
    "KNN": make_pipeline(KNeighborsClassifier()),
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    "Random Forest": make_pipeline(RandomForestClassifier(random_state=RANDOM_STATE)),
}

## 2. Cross-validation on the training set
5-fold **stratified** cross-validation gives a more reliable estimate than a single split and does not touch the test set.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_rows = []
for name, pipe in default_models.items():
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({"Model": name, **{m: s[f"test_{m}"].mean() for m in scoring}})
cv_default = pd.DataFrame(cv_rows).round(4)
cv_default

,Model,accuracy,precision,recall,f1,roc_auc
0,KNN,0.7682,0.5674,0.5324,0.5492,0.7805
1,Logistic Regression,0.8025,0.6541,0.5431,0.5928,0.8460
2,Random Forest,0.7872,0.6349,0.4716,0.5407,0.8222


## 3. Fit on the full training set and evaluate on the test set

In [4]:
rows = []
for name, pipe in default_models.items():
    pipe.fit(X_train, y_train)
    rows.append(evaluate(pipe, X_test, y_test, name, "Default"))

baseline = results_table(rows)
baseline.to_csv("../results/baseline_results.csv", index=False)   # reused in the next notebook
baseline

,Model,Setting,Accuracy,Precision,Recall,F1,ROC-AUC
0,KNN,Default,0.7630,0.5538,0.5508,0.5523,0.7921
1,Logistic Regression,Default,0.8055,0.6582,0.5561,0.6029,0.8422
2,Random Forest,Default,0.7956,0.6473,0.5053,0.5676,0.8258


## 4. Observations
- Accuracy looks similar across models, but **recall** (the share of real churners we catch) is much lower than accuracy. This is the class-imbalance effect seen in the EDA.
- Default models predict "stay" too often because "stay" is the majority class. Tuning and class weighting (next notebook) should help recall and F1.
- The baseline results are saved to `results/baseline_results.csv`.